# SPARC Example 20: Rotation Curve Gallery

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.3**

A gallery of 12 SPARC rotation curves illustrating the full diversity
of shapes found across galaxy types — from rising solid-body curves
in dwarfs to declining Keplerian-like profiles in massive spirals.

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7.1_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.23442703  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/23442703/files/rotation_curve_corpus_v7.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

with open('rotation_curve_corpus_v7.json') as f:
    corpus = json.load(f)

sparc = [g for g in corpus['galaxies']
         if g.get('survey')=='SPARC' and g.get('data') and len(g['data'])>=5]
sparc_sorted = sorted(sparc, key=lambda x: max(p['Vobs'] for p in x['data']))
n = len(sparc_sorted)
indices = [int(i*n/12) for i in range(12)]
selected = [sparc_sorted[i] for i in indices]

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

for ax, g in zip(axes, selected):
    d = g['data']
    R    = [p['Rad']  for p in d]
    Vobs = [p['Vobs'] for p in d]
    vmax = max(Vobs)
    # color by Vmax
    c = plt.cm.plasma(vmax / 300)
    ax.plot(R, Vobs, 'o-', color=c, ms=3, lw=1.2)
    ax.set_title(f"{g['galaxy']}\nVmax={vmax:.0f} km/s", fontsize=8)
    ax.set_xlabel('R (kpc)', fontsize=7)
    ax.set_ylabel('Vrot (km/s)', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.3)

plt.suptitle('HI Corpus v7.3 — SPARC Rotation Curve Gallery\n'
             '12 galaxies spanning the full Vmax range', fontsize=12)
plt.tight_layout()
plt.savefig('ex20_rotation_curve_gallery.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gallery galaxies:')
for g in selected:
    vmax = max(p['Vobs'] for p in g['data'])
    print(f"  {g['galaxy']:<15} Vmax={vmax:.1f} km/s  N_rings={len(g['data'])}")
